In [1]:
# %pip install torch "transformers>=4.45,<5"

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)

   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: C:\Users\adivp\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2",
    attn_implementation="eager",
).to("cpu").float()

model.eval()
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 loaded on:", next(model.parameters()).device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

C:\Users\adivp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\adivp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adivp\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an ad

GPT-2 loaded on: cpu
Parameters: 124,439,808


In [7]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt")

print("Token IDs:", inputs["input_ids"][0].tolist())
print("Token pieces:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.inference_mode():
    outputs = model(**inputs, use_cache=False)

print("Logits shape:", outputs.logits.shape)

next_token_logits = outputs.logits[0, -1, :]
next_token_id = next_token_logits.argmax().item()

print("Next token:", repr(tokenizer.decode([next_token_id])))

Token IDs: [464, 3139, 286, 4881, 318]
Token pieces: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
Logits shape: torch.Size([1, 5, 50257])
Next token: ' the'


In [8]:
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital of the French Republic.

The French Republic


In [11]:
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

The capital of France is in the center, and at a high level. In an interview with Paris Saint-Germain earlier this year, Michel said that there was only one way to get around it: "It must be as simple as possible."
 (See here.) He also pointed out that what he called a


In [12]:
from pathlib import Path

export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

# Explicitly export little-endian float32 values in row-major order.
token_weights = (
    model.transformer.wte.weight.detach().cpu().numpy().astype("<f4")
)

position_weights = (
    model.transformer.wpe.weight.detach().cpu().numpy().astype("<f4")
)

token_weights.tofile(export_dir / "token_embeddings.bin")
position_weights.tofile(export_dir / "position_embeddings.bin")

print("Export folder:", export_dir.resolve())
print("Token embeddings:", token_weights.shape)
print("Position embeddings:", position_weights.shape)

Export folder: C:\Users\adivp\Desktop\inference-engine\exports
Token embeddings: (50257, 768)
Position embeddings: (1024, 768)


In [13]:
token_ids = torch.tensor([[464, 3139, 286, 4881, 318]])
position_ids = torch.arange(token_ids.shape[1]).unsqueeze(0)

with torch.inference_mode():
    reference_x = (
        model.transformer.wte(token_ids)
        + model.transformer.wpe(position_ids)
    )

print("Shape:", reference_x.shape)
print("First token, first 8 embedding values:")
print(reference_x[0, 0, :8].tolist())

Shape: torch.Size([1, 5, 768])
First token, first 8 embedding values:
[-0.08744361251592636, -0.21771633625030518, 0.06849727779626846, -0.05073772370815277, -0.049695029854774475, -0.16734999418258667, -0.1802556961774826, -0.19777514040470123]


In [14]:
ln = model.transformer.h[0].ln_1

ln.weight.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_ln1_weight.bin"
)

ln.bias.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_ln1_bias.bin"
)

with torch.inference_mode():
    reference_norm = ln(reference_x)

print("First token, first 8 normalized values:")
print(reference_norm[0, 0, :8].tolist())

First token, first 8 normalized values:
[-0.052923522889614105, -0.07779036462306976, -0.032439105212688446, -0.028004392981529236, -0.03955240547657013, -0.09703025221824646, 0.13231661915779114, -0.061669960618019104]


In [15]:
projection = model.transformer.h[0].attn.c_attn

projection.weight.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_qkv_weight.bin"
)

projection.bias.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_qkv_bias.bin"
)

with torch.inference_mode():
    reference_qkv = projection(reference_norm)

print("Shape:", reference_qkv.shape)
print("First token, first 8 Q values:")
print(reference_qkv[0, 0, :8].tolist())

Shape: torch.Size([1, 5, 2304])
First token, first 8 Q values:
[0.15319371223449707, -0.41475117206573486, 0.36701321601867676, -1.1779779195785522, 0.1678028255701065, 0.6630426645278931, -0.6845549941062927, 0.01911821961402893]


In [16]:
projection = model.transformer.h[0].attn.c_attn

projection.weight.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_qkv_weight.bin"
)

projection.bias.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_qkv_bias.bin"
)

with torch.inference_mode():
    reference_qkv = projection(reference_norm)

print(reference_qkv[0, 0, :8].tolist())

[0.15319371223449707, -0.41475117206573486, 0.36701321601867676, -1.1779779195785522, 0.1678028255701065, 0.6630426645278931, -0.6845549941062927, 0.01911821961402893]


In [17]:
import math

# Split [1, T, 2304] into three [1, T, 768] tensors.
q_ref, k_ref, v_ref = reference_qkv.split(768, dim=-1)

T = q_ref.shape[1]

# Reorganize into [batch, heads, tokens, head_dim].
q_heads = q_ref.reshape(1, T, 12, 64).transpose(1, 2)
k_heads = k_ref.reshape(1, T, 12, 64).transpose(1, 2)

scores_ref = (q_heads @ k_heads.transpose(-2, -1)) / math.sqrt(64)

future_mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1,
)

scores_ref = scores_ref.masked_fill(future_mask, float("-inf"))

print("Head 0, last token attention scores:")
print(scores_ref[0, 0, -1, :].tolist())

Head 0, last token attention scores:
[-0.5530702471733093, -1.9432626962661743, -2.83024263381958, -2.4178459644317627, -2.5698933601379395]


In [18]:
weights_ref = torch.softmax(scores_ref, dim=-1)

print("Head 0, last token attention weights:")
print(weights_ref[0, 0, -1, :].tolist())

print("Sum:", weights_ref[0, 0, -1, :].sum().item())

Head 0, last token attention weights:
[0.6099011898040771, 0.1518820822238922, 0.06255991011857986, 0.09449257701635361, 0.0811641737818718]
Sum: 0.9999999403953552


In [19]:
# [batch, tokens, 768] -> [batch, heads, tokens, 64]
v_heads = v_ref.reshape(1, T, 12, 64).transpose(1, 2)

# [batch, heads, tokens, tokens] @ [batch, heads, tokens, 64]
context_heads = weights_ref @ v_heads

# Put the heads side by side again.
context_ref = (
    context_heads.transpose(1, 2)
    .contiguous()
    .reshape(1, T, 768)
)

print("Last token, first 8 context values:")
print(context_ref[0, -1, :8].tolist())

Last token, first 8 context values:
[0.00016281665011774749, -0.023537883535027504, 0.09393779188394547, 0.04178185388445854, -0.11379996687173843, 0.041427548974752426, 0.03603730350732803, -0.058491453528404236]


In [20]:
out_projection = model.transformer.h[0].attn.c_proj

out_projection.weight.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_attn_out_weight.bin"
)

out_projection.bias.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_attn_out_bias.bin"
)

with torch.inference_mode():
    attention_output_ref = out_projection(context_ref)
    after_attention_ref = reference_x + attention_output_ref

print("Last token, first 8 values after attention + residual:")
print(after_attention_ref[0, -1, :8].tolist())

Last token, first 8 values after attention + residual:
[-0.2549079656600952, 0.04649011418223381, 0.3931017518043518, -0.1344147026538849, -0.12965092062950134, 0.10563524067401886, -0.05338531732559204, 2.844362497329712]


In [22]:
block = model.transformer.h[0]

parameters = {
    "block0_ln2_weight": block.ln_2.weight,
    "block0_ln2_bias": block.ln_2.bias,
    "block0_fc_weight": block.mlp.c_fc.weight,
    "block0_fc_bias": block.mlp.c_fc.bias,
    "block0_mlp_out_weight": block.mlp.c_proj.weight,
    "block0_mlp_out_bias": block.mlp.c_proj.bias,
}

for name, parameter in parameters.items():
    parameter.detach().cpu().numpy().astype("<f4").tofile(
        export_dir / f"{name}.bin"
    )

with torch.inference_mode():
    mlp_input_ref = block.ln_2(after_attention_ref)
    block_output_ref = after_attention_ref + block.mlp(mlp_input_ref)

print("Last token, first 8 block output values:")
print(block_output_ref[0, -1, :8].tolist())

Last token, first 8 block output values:
[-1.025071382522583, 0.44096606969833374, -0.2954041361808777, -0.1910935938358307, 0.8335590362548828, 0.5234135389328003, -0.06835195422172546, 3.474919080734253]


In [23]:
# Run the original block directly, independently of our manual steps.
with torch.inference_mode():
    direct_block_ref = model.transformer.h[0](
        reference_x,
        use_cache=False,
    )[0]

direct_block_ref.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "block0_reference.bin"
)

In [24]:
for layer, block in enumerate(model.transformer.h):
    parameters = {
        "ln1_weight": block.ln_1.weight,
        "ln1_bias": block.ln_1.bias,
        "qkv_weight": block.attn.c_attn.weight,
        "qkv_bias": block.attn.c_attn.bias,
        "attn_out_weight": block.attn.c_proj.weight,
        "attn_out_bias": block.attn.c_proj.bias,
        "ln2_weight": block.ln_2.weight,
        "ln2_bias": block.ln_2.bias,
        "fc_weight": block.mlp.c_fc.weight,
        "fc_bias": block.mlp.c_fc.bias,
        "mlp_out_weight": block.mlp.c_proj.weight,
        "mlp_out_bias": block.mlp.c_proj.bias,
    }

    for name, parameter in parameters.items():
        parameter.detach().cpu().numpy().astype("<f4").tofile(
            export_dir / f"block{layer}_{name}.bin"
        )

print("Exported all 12 blocks.")

Exported all 12 blocks.


In [25]:
with torch.inference_mode():
    stack_reference = reference_x.clone()

    for block in model.transformer.h:
        stack_reference = block(
            stack_reference,
            use_cache=False,
        )[0]

stack_reference.cpu().numpy().astype("<f4").tofile(
    export_dir / "stack_reference.bin"
)

print("Last token, first 8 values:")
print(stack_reference[0, -1, :8].tolist())

Last token, first 8 values:
[-5.984010696411133, 0.8471170663833618, -2.2699408531188965, 0.9540079236030579, 4.219346046447754, -0.7415431141853333, 0.6376975774765015, 12.589752197265625]


In [26]:
final_ln = model.transformer.ln_f

final_ln.weight.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "final_ln_weight.bin"
)

final_ln.bias.detach().cpu().numpy().astype("<f4").tofile(
    export_dir / "final_ln_bias.bin"
)

# Use the full reference model for an independent comparison.
with torch.inference_mode():
    reference_logits = model(
        input_ids=token_ids,
        use_cache=False,
    ).logits[0, -1]

reference_logits.cpu().numpy().astype("<f4").tofile(
    export_dir / "logits_reference.bin"
)

predicted_id = reference_logits.argmax().item()

print("Reference token ID:", predicted_id)
print("Reference token:", repr(tokenizer.decode([predicted_id])))

Reference token ID: 262
Reference token: ' the'


In [28]:
with torch.inference_mode():
    reference_ids = model.generate(
        input_ids=token_ids,
        attention_mask=torch.ones_like(token_ids),
        max_new_tokens=20,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
    )

print("Reference IDs:", reference_ids[0].tolist())
print(tokenizer.decode(reference_ids[0], skip_special_tokens=True))

Reference IDs: [464, 3139, 286, 4881, 318, 262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 4141, 2066, 318, 262, 3139, 286, 262, 4141]
The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French
